In [1]:
import pdal
import geopandas as gpd
import json
import pandas as pd
import numpy as np
from shapely import MultiPoint
import shapely
from buildingregulariser import regularize_geodataframe
import warnings
import math
import os
from osgeo import gdal
from shapely.geometry import Polygon, MultiPolygon, LinearRing, box
from shapely.ops import unary_union, polygonize
import swifter
# from multiprocessing import Pool, cpu_count
from multiprocessing.dummy import Pool

/Users/fernandogomes/miniconda3/envs/pdal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
## TODO
# Verificar a existência prévia do arquivo

In [2]:
RESULT_FOLDER = '/Users/fernandogomes/dev/LiDAR_produtos'

In [3]:
resolution = 0.5
gdf_distritos = gpd.read_file('data/SIRGAS_GPKG_distrito.gpkg')
gdf_articulacao_17_20 = gpd.read_file("zip://data/SIRGAS_SHP_quadriculamdt.zip!/SIRGAS_SHP_quadriculamdt/")
gdf_articulacao_24 = gpd.read_file('results/folhas_sp_cortada.gpkg')
gdf_articulacao_24.to_crs(epsg=31983, inplace=True)

In [4]:
distritos = [i for _, i in gdf_distritos.iterrows()]

In [5]:
gdf_quadras = gpd.read_file('data/SIRGAS_GPKG_quadraviariaed_2017.gpkg')
so_quadra = gdf_quadras.qe_tipo == 'Quadra'
gdf_quadras = gdf_quadras[so_quadra].reset_index()
gdf_quadras.geometry = gdf_quadras.buffer(2)

In [6]:
agg = {
    'coords':list,  
    'Z':['count', 'median', 'std', 'max'], 
    # 'Intensity':'median', 
    # 'Red':'median',
    # 'Green':'median',
    # 'Blue':'median'  
}

columns = {
    ('coords', 'list'):'coords',
    ('Z', 'count'):'z_count',
    ('Z', 'median'):'z_median',
    ('Z', 'std'):'z_std',
    ('Z', 'max'):'z_max',
    # ('Intensity', 'median'):'intensity_median',
    # ('Red', 'median'):'red_median',
    # ('Green', 'median'):'green_median',
    # ('Blue', 'median'):'blue_median',
}

In [7]:
def nome_distrito(distrito):
    return distrito.ds_nome.replace(' ', '-')

In [8]:
def get_coordenadas(distrito):
    ds = gdal.Open(f"temp/BHM-{nome_distrito(distrito)}.tiff")
    gt = ds.GetGeoTransform()

    cols = ds.RasterXSize
    rows = ds.RasterYSize

    minx = gt[0]
    maxy = gt[3]
    maxx = minx + cols * gt[1]
    miny = maxy + rows * gt[5]

    return [minx, miny, maxx, maxy]

In [9]:
def prepara_folhas(distrito):
    gdf_folhas = gdf_articulacao_24.overlay(gdf_distritos[gdf_distritos.ds_nome == distrito.ds_nome])
    folhas = [f'{RESULT_FOLDER}/2024/BHM/BHM-{nome}-2024-50cm.tiff' for nome in gdf_folhas.nome.to_list()]
    ' '.join(folhas)
    !gdalbuildvrt temp/BHM-{nome_distrito(distrito)}.vrt {' '.join(folhas)} > /dev/null
    !gdal_translate temp/BHM-{nome_distrito(distrito)}.vrt temp/BHM-{nome_distrito(distrito)}.tiff > /dev/null
    quadras = gdf_quadras.overlay(gpd.GeoDataFrame(geometry=[box(*get_coordenadas(distrito))], crs="EPSG:31983"))
    quadras.to_file(f'temp/quadras_viarias-{nome_distrito(distrito)}.gpkg')
    !gdal_rasterize -burn 1.0 -tr 0.5 0.5 -a_nodata 0.0 -te {' '.join([str(x) for x in get_coordenadas(distrito)])} -ot Float32 -of GTiff temp/quadras_viarias-{nome_distrito(distrito)}.gpkg  temp/quadras_viarias-{nome_distrito(distrito)}.tiff > /dev/null
    !gdal_calc.py -A temp/quadras_viarias-{nome_distrito(distrito)}.tiff -B temp/BHM-{nome_distrito(distrito)}.tiff --outfile=temp/BHM-{nome_distrito(distrito)}.tiff --calc="A*B" --NoDataValue=0 --overwrite > /dev/null

In [10]:
# def laz_pipeline(resolution):
#     return [
#         {
#             "type":"readers.las",
#             "filename":f"/Users/fernandogomes/dev/LiDAR_produtos/2024/LiDAR-subs/10-BELEM-buildings-50cm.laz"
#         },
#         {
#             "type":"filters.dbscan",
#             "min_points":5,
#             "eps": 0.5 * math.sqrt(2),
#             "dimensions":"X,Y,Z",
#             "where":"Z > 1."
#             # "eps": 0.5 * 2,
#         },
#     ]

In [11]:
def remove_small_holes(geom, area_threshold):
    if geom is None:
        return None
    
    if geom.geom_type == 'Polygon':
        # Exterior permanece o mesmo
        exterior = geom.exterior
        # Filtra os buracos (interiors) por área
        new_interiors = [
            hole for hole in geom.interiors
            if Polygon(hole).area >= area_threshold
        ]
        return Polygon(exterior, new_interiors)

    elif geom.geom_type == 'MultiPolygon':
        cleaned_polygons = []
        for poly in geom.geoms:
            exterior = poly.exterior
            new_interiors = [
                hole for hole in poly.interiors
                if Polygon(hole).area >= area_threshold
            ]
            cleaned_polygons.append(Polygon(exterior, new_interiors))
        return MultiPolygon(cleaned_polygons)

    else:
        return geom

In [12]:
def laz_pipeline(resolution, distrito):
    return [
        {
            "type":"readers.gdal",
            "filename":f"temp/BHM-{nome_distrito(distrito)}.tiff"
        },
        {
            "type":"filters.ferry",
            "dimensions":"band_1 => Z"
        },
        {
            "type":"filters.dbscan",
            "min_points":5,
            "eps": 0.5 * math.sqrt(2),
            "dimensions":"X,Y,Z",
            "where":"Z > 1."
            # "eps": 0.5 * 2,
        },
        # {
        #     "type":"filters.cluster",
        #     "min_points":5,
        #     "max_points": 50000,
        #     "is3d":True,
        #     "tolerance": 0.5 * math.sqrt(2)
        # }
    ]

In [13]:
# Suprimir warnings de buffer instável
warnings.filterwarnings("ignore", message="divide by zero encountered in buffer")

In [14]:
def processa_distrito(distrito, resolution=resolution):
    print(f'Processando {distrito.ds_nome}')
    prepara_folhas(distrito)
    # resolution = resolution
    laz = laz_pipeline(resolution, distrito)
    pipeline = pdal.Pipeline(json.dumps(laz))
    n_points = pipeline.execute()
    os.remove(f'temp/BHM-{nome_distrito(distrito)}.vrt')
    os.remove(f'temp/BHM-{nome_distrito(distrito)}.tiff')
    os.remove(f'temp/quadras_viarias-{nome_distrito(distrito)}.tiff')
    os.remove(f'temp/quadras_viarias-{nome_distrito(distrito)}.gpkg')
    print(f'Pipeline selected {n_points} points')
    arr = pipeline.arrays[0]
    df = pd.DataFrame(arr)
    df = df[df.ClusterID >= 0]
    df = df[(df.Z > 2.0) & (df.Z < 200.0)]
    df = df[df.groupby("ClusterID")["ClusterID"].transform("count") > 16].reset_index()
    df.loc[:, 'coords'] = list(np.dstack([df.X, df.Y])[0])
    df['Z'] = df.groupby(['X', 'Y'])['Z'].transform('max')
    df.drop_duplicates(subset=['X', 'Y'], keep='last', inplace=True)
    df_agg = df.groupby('ClusterID').agg(agg)
    df_agg.columns = df_agg.columns.to_flat_index()
    df_agg.rename(columns=columns, inplace=True)
    df_agg.loc[:, 'geometry'] = df_agg.coords.apply(MultiPoint)
    gdf_agg = gpd.GeoDataFrame(df_agg)
    gdf_agg.set_crs(epsg=31983, inplace=True)
    gdf_agg.drop(columns=['coords'])
    mask = gdf_agg['z_count'] >= 16
    # gdf_agg.loc[mask, 'geometry'] = gdf_agg.loc[mask, 'geometry'].swifter.apply(lambda x: shapely.concave_hull(x, ratio=0.1, allow_holes=True))
    gdf_agg.loc[mask, 'geometry'] = gdf_agg.loc[mask, 'geometry'].apply(lambda x: shapely.concave_hull(x, ratio=0.1, allow_holes=True))
    gdf_agg["geometry"] = gdf_agg["geometry"].buffer(0)
    gdf_regularizado = regularize_geodataframe(
        gdf_agg[gdf_agg.area > 0].reset_index(),
        parallel_threshold=1.,
        simplify_tolerance=1.,
        allow_45_degree=True,
        diagonal_threshold_reduction=0.5,
        neighbor_alignment=False
        )
    gdf_regularizado["geometry"] = gdf_regularizado["geometry"].apply(lambda geom: remove_small_holes(geom, area_threshold=0.5))
    gdf_regularizado = gdf_regularizado.overlay(gdf_distritos[gdf_distritos.ds_nome == distrito.ds_nome])
    gdf_regularizado.loc[:, 'area_de_projecao'] = gdf_regularizado.area
    gdf_regularizado.loc[:, 'contagem_de_pixels'] = gdf_regularizado.loc[:, 'z_count']
    gdf_regularizado.loc[:, 'gabarito'] = gdf_regularizado.loc[:, 'z_median']
    gdf_regularizado.loc[:, 'pavimentos'] = gdf_regularizado.loc[:, 'z_median'] // 3.4
    gdf_regularizado.loc[gdf_regularizado.z_max < 3.4, 'pavimentos'] = 1.0
    gdf_regularizado.loc[:, 'area_total_construida'] = gdf_regularizado.loc[:, 'pavimentos'] * gdf_regularizado.loc[:, 'area_de_projecao']
    colunas_manter = ['area_de_projecao', 'gabarito', 'pavimentos', 'area_total_construida', 'contagem_de_pixels', 'geometry']
    gdf_regularizado.loc[gdf_regularizado.area > 1.5, colunas_manter].reset_index().to_file(f'results/distritos/{nome_distrito(distrito)}-multipoligono.gpkg', driver='GPKG')


In [15]:
n_processos = 12

# Cria o pool e distribui os dados
with Pool(n_processos) as pool:
    resultados = pool.map(processa_distrito, distritos)


Processando MANDAQUIProcessando ARTUR ALVIM

Processando ITAIM BIBI
Processando JAGUARA
Processando JOSE BONIFACIO
Processando LAPA
Processando LIMAO
Processando CURSINO
Processando VILA MARIANA
Processando TUCURUVI
Processando SAPOPEMBA
Processando JABAQUARA
Pipeline selected 74579956 points
Pipeline selected 79420476 points
Pipeline selected 120656965 points
Pipeline selected 139955004 points
Pipeline selected 99124745 points
Pipeline selected 118734458 points
Pipeline selected 79956282 points
Pipeline selected 100017600 points
Pipeline selected 99666682 points
Pipeline selected 135964796 points
Pipeline selected 148640440 points
Pipeline selected 177664888 points
Processando JARDIM SAO LUIS
Pipeline selected 225531282 points
Processando SAUDE
Pipeline selected 99135380 points
Processando MOEMA
Pipeline selected 90245260 points
Processando LAJEADO
Pipeline selected 119264599 points
Processando ITAIM PAULISTA
Pipeline selected 124337664 points
Processando SAO LUCAS
Processando LIBERDA

IOStream.flush timed out


Pipeline selected 119627340 points
Pipeline selected 173494112 points
Processando SACOMA
Pipeline selected 171958821 points
Processando VILA ANDRADE
Pipeline selected 99440467 points
Processando VILA FORMOSA
Pipeline selected 99271000 points
Processando VILA JACUI
Pipeline selected 79956972 points
Processando VILA MARIA
Pipeline selected 100112985 points
Processando REPUBLICA
Pipeline selected 44999396 points
Processando TATUAPE
Pipeline selected 99358428 points
Processando VILA MEDEIROS
Pipeline selected 99395620 points
Processando VILA CURUCA
Pipeline selected 80092512 points
Processando VILA SONIA
Pipeline selected 124374354 points
Processando ALTO DE PINHEIROS
Pipeline selected 99441604 points
Processando JAGUARE
Pipeline selected 79742352 points
Processando JARDIM PAULISTA
Pipeline selected 99577840 points
Processando PARELHEIROS
Pipeline selected 1679983877 points
Processando VILA GUILHERME
Pipeline selected 79376264 points
Processando VILA LEOPOLDINA
Pipeline selected 79473920 p

IOStream.flush timed out


Pipeline selected 119257950 points
Processando CASA VERDE
Processando CIDADE LIDER
Pipeline selected 79670892 points
Pipeline selected 119601104 points
Processando ERMELINO MATARAZZO
Pipeline selected 99528030 points
Processando GRAJAU
Pipeline selected 694008000 points
Processando ANHANGUERA
Pipeline selected 313445116 points
Processando SANTANA
Pipeline selected 100193233 points
Processando JARDIM ANGELA
Pipeline selected 358111889 points
Processando SAO DOMINGOS
Pipeline selected 123786036 points
Processando BELEM
Pipeline selected 79659720 points
Processando FREGUESIA DO O
Pipeline selected 124225092 points
Processando BOM RETIRO
Pipeline selected 59858360 points
Processando CIDADE DUTRA


IOStream.flush timed out


Pipeline selected 355139320 points
Processando CARRAO
Pipeline selected 99049000 points
Processando BRASILANDIA
Pipeline selected 236039332 points
Processando CONSOLACAO
Pipeline selected 45005830 points
Processando SAO MATEUS
Pipeline selected 148524660 points
Processando BELA VISTA
Pipeline selected 59603652 points
Processando CACHOEIRINHA
Pipeline selected 161302168 points
Processando SANTO AMARO
Processando BRAS
Pipeline selected 45167252 points
Pipeline selected 149373730 points
Processando CAMPO GRANDE
Pipeline selected 149127462 points
Processando PARQUE DO CARMO
Pipeline selected 177514656 points
Processando BUTANTA
Pipeline selected 148827893 points
Processando CAMBUCI
Pipeline selected 59861646 points


KeyboardInterrupt: 